# Annual HP–LA spatial clustering test

This notebook tests whether annual binary High Pressure–Low Accessibility (HP–LA) LSOAs are more spatially clustered than expected under random label assignment. It uses the already-derived Travel-time 30-minute HP–LA classifications and does not recalculate E2SFCA accessibility, annual thresholds, trajectories, or bivariate LISA.

Because HP–LA is binary, the primary statistic is the global **BB join count**: the number of neighbouring pairs for which both LSOAs are HP–LA. Inference uses 999 permutations separately for 2001, 2011 and 2021.

## Reproducibility contract

**Inputs.** The fixed 2021 LSOA boundary and the three annual HP–LA status files created by Step 4. Set `DISSERTATION_DATA_ROOT`; optional overrides are `HPLA_RESULTS_DIR`, `LSOA_BOUNDARY_GPKG`, and `GLOBAL_BB_OUTPUT_DIR`.

**Method.** Binary fuzzy-contiguity weights (`intersects`, no buffer), 999 permutations and fixed year-specific random seeds. Islands are reported and excluded from the join-count statistic.

**Outputs.** Annual observed/expected BB joins, permutation intervals and pseudo-p values, plus spatial-weights, input, run and output manifests. The test establishes global clustering only; it does not locate significant clusters or estimate causality.


## 1. Imports and fixed configuration

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from pathlib import Path

import esda
import geopandas as gpd
import libpysal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from esda.join_counts import Join_Counts
from libpysal.weights import fuzzy_contiguity, w_subset

RUN_DIR = Path.cwd().resolve()
REPOSITORY_ROOT = RUN_DIR.parents[1]
DATA_ROOT = Path(os.environ['DISSERTATION_DATA_ROOT']).expanduser().resolve()
SOURCE_DIR = Path(
    os.environ.get(
        'HPLA_RESULTS_DIR',
        DATA_ROOT / 'final_data_and_analysis/Downstream_Analysis_Descriptive_E2SFCA_Longitudinal_20260820/04_TravelTime_E2SFCA_30min_ExactHalo20_40',
    )
).expanduser().resolve()
BOUNDARY_GPKG = Path(
    os.environ.get(
        'LSOA_BOUNDARY_GPKG',
        DATA_ROOT / 'final_data_and_analysis/unified-lsoa/common2021_lsoa_spatial_foundation.gpkg',
    )
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get('GLOBAL_BB_OUTPUT_DIR', REPOSITORY_ROOT / '_private_outputs' / 'global_bb')
).expanduser().resolve()
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
QA_DIR = OUTPUT_DIR / 'qa'
for directory in (TABLE_DIR, FIGURE_DIR, QA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

YEARS = (2001, 2011, 2021)
EXPECTED_LSOAS = 3411
PERMUTATIONS = 999
RANDOM_SEED = 20260823
HP_LA_FIELD = 'within_year_relative_hpla'
STATUS_PATHS = {year: SOURCE_DIR / 'tables' / f'annual_hp_la_status_{year}.csv' for year in YEARS}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print('Output directory:', OUTPUT_DIR)
print('Statistic: global binary BB join count')
print('Permutations per year:', PERMUTATIONS)

## 2. Input validation and binary spatial weights

The formal adjacency structure matches the existing spatial-association implementation: PySAL fuzzy contiguity with polygon intersection and no buffering. Join counts use binary, not row-standardised, weights. The single island is retained in the source data but excluded from the statistic because it has no neighbour joins.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def as_binary(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype('int8')
    converted = series.astype(str).str.strip().str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0})
    assert converted.notna().all(), 'HP–LA field contains values other than True/False or 1/0'
    return converted.astype('int8')

required_inputs = [BOUNDARY_GPKG, *STATUS_PATHS.values()]
missing = [str(path) for path in required_inputs if not path.is_file()]
assert not missing, f'Missing required inputs: {missing}'

geometry = gpd.read_file(BOUNDARY_GPKG, layer='lsoa_2021').to_crs('EPSG:27700')
geometry['lsoa_code'] = geometry['lsoa_code'].astype(str)
assert len(geometry) == EXPECTED_LSOAS
assert geometry['lsoa_code'].is_unique

indexed_geometry = geometry.set_index('lsoa_code', drop=False)
w_all = fuzzy_contiguity(
    indexed_geometry,
    predicate='intersects',
    buffering=False,
    silence_warnings=True,
)
w_all.transform = 'b'
islands = list(w_all.islands)
connected_ids = [lsoa for lsoa in w_all.id_order if lsoa not in islands]
w = w_subset(w_all, connected_ids, silence_warnings=True)
w.transform = 'b'
order = list(w.id_order)
assert w.n == EXPECTED_LSOAS - len(islands)
assert not w.islands

status_frames = {}
for year, path in STATUS_PATHS.items():
    frame = pd.read_csv(path, encoding='utf-8-sig', dtype={'lsoa_code': str})
    assert len(frame) == EXPECTED_LSOAS
    assert frame['lsoa_code'].is_unique
    assert frame['year'].eq(year).all()
    assert set(frame['lsoa_code']) == set(geometry['lsoa_code'])
    frame[HP_LA_FIELD] = as_binary(frame[HP_LA_FIELD])
    status_frames[year] = frame

cardinalities = np.asarray(list(w_all.cardinalities.values()), dtype=int)
weights_audit = pd.DataFrame([{
    'total_lsoas': EXPECTED_LSOAS,
    'lsoas_in_join_count': w.n,
    'undirected_links': int(w_all.sparse.nnz // 2),
    'mean_neighbours': float(w_all.mean_neighbors),
    'median_neighbours': float(np.median(cardinalities)),
    'maximum_neighbours': int(cardinalities.max()),
    'island_count': len(islands),
    'islands': ' | '.join(islands),
    'weight_construction': 'PySAL fuzzy_contiguity: intersects, no buffer',
    'weight_transform': 'binary',
}])
weights_audit.to_csv(QA_DIR / 'spatial_weights_audit.csv', index=False)

input_manifest = pd.DataFrame([
    {'role': 'fixed_2021_lsoa_geometry', 'path': str(BOUNDARY_GPKG), 'sha256': sha256(BOUNDARY_GPKG)},
    *[{'role': f'annual_hp_la_status_{year}', 'path': str(path), 'sha256': sha256(path)} for year, path in STATUS_PATHS.items()],
])
input_manifest.to_csv(QA_DIR / 'input_manifest.csv', index=False)

display(weights_audit)

## 3. Annual global BB join-count tests

For each year, HP–LA is coded 1 and all other LSOAs 0. The one-sided permutation pseudo-p value tests whether the observed number of HP–LA–HP–LA neighbour pairs is unusually high.

In [ ]:
rows = []
simulation_distributions = {}

for year in YEARS:
    frame = status_frames[year].set_index('lsoa_code')
    y = frame.loc[order, HP_LA_FIELD].astype(int).to_numpy()
    np.random.seed(RANDOM_SEED + year)
    join_count = Join_Counts(y, w, permutations=PERMUTATIONS, drop_islands=True)
    simulated_bb = np.asarray(join_count.sim_bb, dtype=float)
    simulation_distributions[year] = simulated_bb
    ci_low, ci_high = np.quantile(simulated_bb, [0.025, 0.975])
    hp_la_total = int(status_frames[year][HP_LA_FIELD].sum())
    rows.append({
        'year': year,
        'n_lsoas_total': EXPECTED_LSOAS,
        'n_lsoas_in_test': w.n,
        'hp_la_lsoas': hp_la_total,
        'hp_la_share_percent': 100 * hp_la_total / EXPECTED_LSOAS,
        'total_neighbour_pairs': float(join_count.J),
        'observed_bb': float(join_count.bb),
        'permutation_mean_bb': float(join_count.mean_bb),
        'permutation_ci_2_5': float(ci_low),
        'permutation_ci_97_5': float(ci_high),
        'bb_excess_over_mean': float(join_count.bb - join_count.mean_bb),
        'bb_observed_expected_ratio': float(join_count.bb / join_count.mean_bb),
        'p_sim_bb_one_sided': float(join_count.p_sim_bb),
        'observed_bw': float(join_count.bw),
        'permutation_mean_bw': float(join_count.mean_bw),
        'permutations': PERMUTATIONS,
        'random_seed': RANDOM_SEED + year,
    })

results = pd.DataFrame(rows)
results['significant_positive_clustering_0_05'] = (
    (results['observed_bb'] > results['permutation_ci_97_5'])
    & (results['p_sim_bb_one_sided'] < 0.05)
)
results.to_csv(TABLE_DIR / 'annual_hp_la_global_bb_join_counts.csv', index=False)
display(results.round(3))

## 4. Figure: observed versus permutation-expected BB joins

In [ ]:
x = np.arange(len(results))
fig, ax = plt.subplots(figsize=(7.6, 4.8))

ax.vlines(
    x,
    results['permutation_ci_2_5'],
    results['permutation_ci_97_5'],
    color='#bdbdbd',
    linewidth=8,
    label='95% permutation interval',
    zorder=1,
)
ax.scatter(
    x, results['permutation_mean_bb'],
    s=55, color='#525252', marker='o',
    label='Permutation mean', zorder=2,
)
ax.scatter(
    x, results['observed_bb'],
    s=78, color='#b2182b', marker='D',
    label='Observed BB joins', zorder=3,
)

offset = max(8.0, 0.025 * results['observed_bb'].max())
for xx, row in results.iterrows():
    ax.text(
        xx, row['observed_bb'] + offset,
        f"p = {row['p_sim_bb_one_sided']:.3f}",
        ha='center', va='bottom', fontsize=9, color='#7f0000',
    )

ax.set_xticks(x, results['year'].astype(int))
ax.set_xlabel('Census year', fontsize=11)
ax.set_ylabel('HP–LA–HP–LA neighbour pairs (BB joins)', fontsize=11)
ax.set_title('Spatial clustering of annual HP–LA LSOAs', fontsize=12, pad=10)
ax.tick_params(axis='both', labelsize=9.5)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='0.9', linewidth=0.7, zorder=0)
ax.legend(frameon=False, fontsize=8.5, loc='center left', bbox_to_anchor=(0.02, 0.53))
fig.tight_layout()

png_path = FIGURE_DIR / 'annual_hp_la_global_bb_join_counts.png'
pdf_path = FIGURE_DIR / 'annual_hp_la_global_bb_join_counts.pdf'
fig.savefig(png_path, dpi=400, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Reporting summary and output manifest

Interpretation is restricted to spatial clustering of the already-derived annual HP–LA category. It does not test causality, unmet need, or the cross-variable neighbouring association previously examined by bivariate LISA.

In [ ]:
for row in results.itertuples(index=False):
    conclusion = 'positive spatial clustering' if row.significant_positive_clustering_0_05 else 'no significant positive clustering'
    print(
        f"{row.year}: observed BB = {row.observed_bb:.0f}; "
        f"permutation mean = {row.permutation_mean_bb:.1f}; "
        f"95% interval = [{row.permutation_ci_2_5:.1f}, {row.permutation_ci_97_5:.1f}]; "
        f"p = {row.p_sim_bb_one_sided:.3f} — {conclusion}."
    )

run_manifest = {
    'analysis': 'Annual HP-LA global binary BB join-count clustering test',
    'source_specification': 'Travel-time-based 30-minute E2SFCA',
    'hp_la_field': HP_LA_FIELD,
    'years': list(YEARS),
    'permutations': PERMUTATIONS,
    'weight_construction': 'PySAL fuzzy_contiguity predicate=intersects, buffering=False',
    'weight_transform': 'binary',
    'islands_excluded_from_join_test': islands,
    'python': sys.version,
    'platform': platform.platform(),
    'package_versions': {
        'esda': esda.__version__,
        'geopandas': gpd.__version__,
        'libpysal': libpysal.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
}
(QA_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

output_paths = [
    TABLE_DIR / 'annual_hp_la_global_bb_join_counts.csv',
    QA_DIR / 'spatial_weights_audit.csv',
    QA_DIR / 'input_manifest.csv',
    QA_DIR / 'run_manifest.json',
    png_path, pdf_path,
]
output_manifest = pd.DataFrame([
    {'path': str(path.relative_to(OUTPUT_DIR)), 'bytes': path.stat().st_size, 'sha256': sha256(path)}
    for path in output_paths
])
output_manifest.to_csv(QA_DIR / 'output_manifest.csv', index=False)
display(output_manifest)